# 05 (v1.1). RAG 형식 LoRA 학습 + 출처 불분명 문단 감점

04 챗봇(검색 → Qwen2.5-7B)에서 **생성 모델을 RAG 형식으로 학습**시키고, **출처 등급이 '기타'인 문단**의 순위를 낮춥니다.

```
train 건강 질문 ─▶ 같은 검색기로 상위 5개 문단 ─▶ 정답 문단이 있으면  "정답 [번호]"
                                              └ 없으면              "거절 문장"
                 ─▶ Qwen2.5-7B-Instruct에 LoRA 학습 (프롬프트는 04와 똑같이)
```

| 단계 | 내용 |
|---|---|
| 3. 검색기 | 04와 같음 + `TIER_PENALTY`: 출처 등급 '기타' 문단의 재정렬 점수에서 빼는 값 |
| 4. 감점 크기 | **검증(val) 200개**로만 정함: 근거 R@5가 거의 안 떨어지는(≤ 0.01) 가장 큰 감점 |
| 5. 학습 데이터 | MS MARCO v1.1 **train**의 건강 질문 (평가·검증은 dev에서 왔으므로 겹치지 않음) |
| 6. LoRA 학습 | bf16 LoRA (r=16, 모든 선형층), 1 epoch. 체크포인트를 드라이브에 저장해 끊겨도 이어서 학습 |
| 7. 평가 | 04와 같은 질문(건강 100 + 분야 밖 50). **원본 Qwen** vs **LoRA** vs **LoRA + 출처 감점** |

**런타임: A100.** 위에서부터 Run all. 전체 약 1시간 (`N_TRAIN=3000` 기준, 시간이 없으면 줄이세요).
결과는 드라이브 `rag_msmarco/v11_test/lora/`에 저장되고, 마지막 셀이 런타임을 끕니다.

**알아둘 점**: MS MARCO 정답은 짧은 구절이라, 학습 후에는 답변이 짧아집니다 (단어 겹침 점수에는 유리, 챗봇 설명은 짧아짐).

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft datasets
!pip -q uninstall -y torchao    # Colab 기본 torchao(0.10)가 최신 peft와 충돌해서 제거

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/v11_test'    # T2 (v1.1) 결과 폴더
OUT_DIR  = f'{SAVE_DIR}/lora'
os.makedirs(OUT_DIR, exist_ok=True)
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)

## 2. 설정

시간이 밀리면 `N_TRAIN`부터 줄이세요 (2000이면 학습 약 25분).

In [ ]:
import re, json, time, random
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
N_TEST     = 100        # 평가할 건강 질문 수 (04와 같게)
GEN_BATCH  = 16
MAX_NEW    = 256

N_TRAIN       = 3000    # 학습 예시 수
REFUSE_FRAC   = 0.15    # 학습 예시 중 거절 예시 비율 (정답 문단이 상위 5개에 없는 질문)
MAX_LEN       = 1024    # 이보다 긴 예시는 버림
EPOCHS        = 1
LR            = 2e-4
BATCH, GRAD_ACC = 8, 2  # 실제 배치 16
LORA_R, LORA_ALPHA = 16, 32

PENALTIES  = [0, 1, 2, 3, 5]   # '기타' 출처 감점 후보 (재정렬 점수 단위)

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. 검색기 (04와 같음 + 출처 감점)

`penalty`만큼 '기타' 등급 문단의 재정렬 점수를 깎은 뒤 다시 줄 세웁니다. 문단을 DB에서 지우지는 않으므로, 의료 사이트에 답이 없으면 '기타' 문단도 여전히 쓰입니다.
거절 기준(`THRESH`)은 감점 전 원래 최고 점수로 판단합니다.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
assert all('gold_pids' in q for q in val_set), 'val.jsonl에 gold_pids가 없습니다'
print(f'문단 {len(corpus):,} / 평가 {len(eval_set)} / 검증 {len(val_set)}')

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

TIERS = {
    '공공기관':   ['medlineplus.gov', 'cdc.gov', 'nih.gov', 'nhs.uk', 'who.int', 'fda.gov', 'cancer.gov'],
    '병원·학회':  ['mayoclinic.org', 'clevelandclinic.org', 'hopkinsmedicine.org', 'aafp.org', 'merckmanuals.com',
                  'kidshealth.org', 'health.harvard.edu', 'heart.org', 'cancer.org', 'familydoctor.org', 'uptodate.com'],
    '상업 매체':  ['webmd.com', 'healthline.com', 'verywell.com', 'medicinenet.com', 'everydayhealth.com', 'drugs.com',
                  'rxlist.com', 'emedicinehealth.com', 'medscape.com', 'medicalnewstoday.com', 'rightdiagnosis.com',
                  'patient.info', 'labtestsonline.org'],
    '커뮤니티·생활': ['healthtap.com', 'sharecare.com', 'livestrong.com', 'babycenter.com', 'psychologytoday.com'],
    '사전':       ['thefreedictionary.com'],
}

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def tier_of(site):
    for tier, sites in TIERS.items():
        if any(site == s or site.endswith('.' + s) for s in sites):
            return tier
    return '기타'

sites   = [site_of(c['url']) for c in corpus]
tiers   = [tier_of(s) for s in sites]
unknown = np.array([t == '기타' for t in tiers])
print('DB 출처 등급:', Counter(tiers).most_common())

norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def retrieve_many(queries, dedup=True, top_n=TOP_N, penalty=0.0):
    # 질문마다 [{pid, score, raw, site, tier, url, text}, ...] 상위 top_n개. score = raw - (기타면 penalty)
    _, ids = index.search(encode(queries), K)
    pairs = [(q, texts[p]) for q, cands in zip(queries, ids.tolist()) for p in cands]
    all_raw = reranker.predict(pairs, batch_size=512, show_progress_bar=len(queries) > 500).reshape(len(queries), K)
    out = []
    for cands, raw in zip(ids.tolist(), all_raw):
        cands = np.array(cands)
        scores = raw - penalty * unknown[cands]
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = int(cands[i])
            key = norm_text(texts[p])
            if dedup and key in seen:
                continue
            seen.add(key)
            docs.append({'pid': p, 'score': float(scores[i]), 'raw': float(raw[i]), 'top_raw': float(raw.max()),
                         'site': sites[p], 'tier': tiers[p], 'url': corpus[p]['url'], 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

## 4. 감점 크기 정하기 (검증 질문으로만)

감점이 클수록 '기타' 문단이 상위 5개에서 빠지지만, 정답 문단이 '기타'일 때는 근거를 놓칩니다.
**근거 R@5가 감점 0 대비 0.01 넘게 떨어지지 않는 가장 큰 감점**을 고릅니다. 시험 질문은 보지 않습니다.

In [ ]:
val_q = [q['query'] for q in val_set]
rows = []
for pen in PENALTIES:
    D = retrieve_many(val_q, penalty=pen)
    rows.append({'penalty': pen,
                 'R@5': np.mean([bool(set(q['gold_pids']) & {d['pid'] for d in docs}) for q, docs in zip(val_set, D)]),
                 "상위5 중 '기타' 비율": np.mean([d['tier'] == '기타' for docs in D for d in docs])})
import pandas as pd
pen_table = pd.DataFrame(rows).set_index('penalty').round(3)
display(pen_table)

base_r5 = pen_table.loc[0, 'R@5']
TIER_PENALTY = max(p for p in PENALTIES if pen_table.loc[p, 'R@5'] >= base_r5 - 0.01)
THRESH = float(np.percentile([docs[0]['top_raw'] for docs in retrieve_many(val_q)], 5))   # 04와 같은 거절 기준
print(f'선택한 감점: {TIER_PENALTY} / 거절 기준(검증 하위 5%): {THRESH:.2f}')

## 5. 학습 데이터 만들기 (train 건강 질문)

- 건강 질문 선별은 T1과 같은 규칙 (문단 10개 중 의료 사이트 2개 이상).
- train 문단은 이미 DB에 들어 있으므로, **04와 같은 검색기**로 상위 5개를 뽑아 실제 사용 때와 같은 입력을 만듭니다.
- 정답 문단(is_selected)이 상위 5개 안에 있으면 → `정답 구절 [번호]`, 없으면 → 거절 문장 (전체의 `REFUSE_FRAC`만큼만).

In [ ]:
from datasets import load_dataset

HEALTH_SITES = [
    'webmd.com', 'mayoclinic.org', 'medicinenet.com', 'medical-dictionary.thefreedictionary.com',
    'verywell.com', 'healthline.com', 'drugs.com', 'livestrong.com', 'healthtap.com', 'cdc.gov',
    'rightdiagnosis.com', 'emedicinehealth.com', 'medscape.com', 'everydayhealth.com',
    'medlineplus.gov', 'nhs.uk', 'rxlist.com', 'sharecare.com', 'medicalnewstoday.com',
    'kidshealth.org', 'nih.gov', 'patient.info', 'labtestsonline.org', 'aafp.org',
    'clevelandclinic.org', 'hopkinsmedicine.org', 'healthgrades.com', 'familydoctor.org',
    'merckmanuals.com', 'uptodate.com', 'health.harvard.edu', 'babycenter.com', 'psychologytoday.com',
]
MIN_HEALTH_HITS = 2
NO_ANSWER = 'No Answer Present.'

def is_health(url):
    h = site_of(url)
    return any(h == s or h.endswith('.' + s) for s in HEALTH_SITES)

clean_query = lambda q: re.sub(r'^[\s.,;:!?\-]+', '', ' '.join(q.split()))
clean_text  = lambda t: ' '.join(t.split())
pid_of = {t: i for i, t in enumerate(texts)}
held_out = {q['qid'] for q in eval_set} | {q['qid'] for q in val_set}

train = load_dataset('microsoft/ms_marco', 'v1.1')['train']
pool = []
for b in train.iter(batch_size=5000):
    for j in range(len(b['query'])):
        p, qid = b['passages'][j], int(b['query_id'][j])
        if qid in held_out or sum(is_health(u) for u in p['url']) < MIN_HEALTH_HITS:
            continue
        answers = [clean_text(a) for a in b['answers'][j] if clean_text(a) and a != NO_ANSWER]
        gold = {pid_of[clean_text(t)] for t, s in zip(p['passage_text'], p['is_selected'])
                if s == 1 and clean_text(t) in pid_of}
        if answers and gold:
            pool.append({'qid': qid, 'query': clean_query(b['query'][j]), 'answer': answers[0], 'gold': gold})
print(f'train 건강 질문 (정답·정답 문단 있음): {len(pool):,}개')

random.Random(SEED).shuffle(pool)
pool = pool[:int(N_TRAIN * 1.6)]                      # 거절 예시 비율을 맞출 여유분까지 검색
t = time.time()
pool_docs = retrieve_many([q['query'] for q in pool])
print(f'검색 {len(pool):,}개: {time.time() - t:.0f}초')

In [ ]:
REFUSAL = "Sorry, I can't answer that from the medical documents I have."

def cite(answer, nums):
    tag = ''.join(f'[{n}]' for n in nums)
    a = answer.strip()
    return a[:-1] + f' {tag}.' if a.endswith('.') else f'{a} {tag}'

answerable, unanswerable = [], []
for q, docs in zip(pool, pool_docs):
    nums = [i for i, d in enumerate(docs, 1) if d['pid'] in q['gold']]
    ex = {'qid': q['qid'], 'query': q['query'], 'pids': [d['pid'] for d in docs]}
    if nums:
        answerable.append({**ex, 'target': cite(q['answer'], nums)})
    else:
        unanswerable.append({**ex, 'target': REFUSAL})

n_ref = min(len(unanswerable), int(N_TRAIN * REFUSE_FRAC))
train_ex = answerable[:N_TRAIN - n_ref] + unanswerable[:n_ref]
random.Random(SEED).shuffle(train_ex)
save_jsonl(train_ex, f'{OUT_DIR}/train_examples.jsonl')
print(f'정답 문단이 상위 5개에 있음 {len(answerable):,} / 없음 {len(unanswerable):,}')
print(f'학습 예시 {len(train_ex):,}개 (답변 {len(train_ex) - n_ref:,} + 거절 {n_ref:,})')
for ex in train_ex[:3]:
    print(' -', ex['query'], '→', ex['target'][:150])

## 6. LoRA 학습

- 프롬프트(시스템 지시 + 문서 [1]~[5] + 질문)는 04와 **똑같이** 만들고, 답변 부분만 손실을 계산합니다.
- 체크포인트는 드라이브 `lora/ckpt/`에 저장됩니다. 런타임이 끊기면 다시 Run all 하면 이어서 학습합니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
llm.config.use_cache = False
llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
llm.enable_input_require_grads()
llm = get_peft_model(llm, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type='CAUSAL_LM',
                                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                                     'gate_proj', 'up_proj', 'down_proj']))
llm.print_trainable_parameters()

def to_features(ex):
    docs = [{'site': sites[p], 'text': texts[p]} for p in ex['pids']]
    prompt = tok.apply_chat_template(rag_messages(ex['query'], docs), tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    a_ids = tok(ex['target'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    return {'input_ids': p_ids + a_ids, 'labels': [-100] * len(p_ids) + a_ids}

feats = [f for f in map(to_features, train_ex) if len(f['input_ids']) <= MAX_LEN]
print(f'학습 예시 {len(feats):,}개 / 평균 길이 {np.mean([len(f["input_ids"]) for f in feats]):.0f} 토큰')

def collate(batch):
    n = max(len(f['input_ids']) for f in batch)
    pad = lambda xs, v: [x + [v] * (n - len(x)) for x in xs]
    ids = pad([f['input_ids'] for f in batch], tok.pad_token_id)
    return {'input_ids': torch.tensor(ids),
            'attention_mask': torch.tensor(pad([[1] * len(f['input_ids']) for f in batch], 0)),
            'labels': torch.tensor(pad([f['labels'] for f in batch], -100))}

In [ ]:
CKPT_DIR = f'{OUT_DIR}/ckpt'
total_steps = -(-len(feats) // (BATCH * GRAD_ACC)) * EPOCHS
args = TrainingArguments(            # transformers 5: warmup_ratio 없음 → warmup_steps로 3%
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
    warmup_steps=max(1, int(total_steps * 0.03)),
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, bf16=True,
    logging_steps=10, save_steps=50, save_total_limit=2, report_to='none',
    remove_unused_columns=False, seed=SEED)
trainer = Trainer(model=llm, args=args, train_dataset=feats, data_collator=collate)

has_ckpt = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
t = time.time()
trainer.train(resume_from_checkpoint=has_ckpt or None)
print(f'학습 끝: {(time.time() - t) / 60:.0f}분')

ADAPTER_DIR = f'{OUT_DIR}/adapter'
llm.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
pd.DataFrame([l for l in trainer.state.log_history if 'loss' in l]).to_csv(f'{OUT_DIR}/train_log.csv', index=False)
print('LoRA 저장:', ADAPTER_DIR)

## 7. 평가: 원본 Qwen vs 문단 10개 vs LoRA vs LoRA + 출처 감점

04의 **RAG 개선**(중복 제거 + 거절 기준)과 같은 조건에서 비교합니다. **원본 Qwen, 문단 10개**는 LLM에 넣는 문단 수만 5 → 10으로 늘린 것입니다 (근거 R@5 0.61을 올리는 가장 간단한 방법). 질문은 04와 같은 건강 100개 + 분야 밖 50개입니다.
원본 Qwen은 LoRA를 잠시 끈 상태(`disable_adapter`)로 생성합니다.

In [ ]:
import string

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)                          # 출처 번호는 채점에서 제외
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)
has_cite = lambda a: bool(re.search(r'\[\d\]', a))

In [ ]:
llm.eval()
llm.config.use_cache = True
tok.padding_side = 'left'

@torch.inference_mode()
def generate(msgs_list, bs=GEN_BATCH):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

test_h = random.Random(SEED).sample([q for q in eval_set if q['domain'] == 'health'], N_TEST)   # 04와 같은 100개
test_o = [q for q in eval_set if q['domain'] != 'health']
qs = test_h + test_o
queries = [q['query'] for q in qs]
print(f'평가: 건강 {len(test_h)}개 + 분야 밖 {len(test_o)}개')

def run(penalty, use_lora, top_n=TOP_N):
    t = time.time()
    docs_all = retrieve_many(queries, penalty=penalty, top_n=top_n)
    answers = [REFUSAL] * len(qs)                     # 관련 문서 없음 → LLM 안 부르고 거절
    todo = [i for i, d in enumerate(docs_all) if d[0]['top_raw'] >= THRESH]
    msgs = [rag_messages(queries[i], docs_all[i]) for i in todo]
    if use_lora:
        gen = generate(msgs)
    else:
        with llm.disable_adapter():
            gen = generate(msgs)
    for i, a in zip(todo, gen):
        answers[i] = a
    return {'answers': answers, 'docs': docs_all, 'sec': (time.time() - t) / len(qs)}

runs = {}
for name, pen, lora, n in [('원본 Qwen', 0, False, TOP_N), ('원본 Qwen, 문단 10개', 0, False, 10),
                           ('LoRA', 0, True, TOP_N), ('LoRA + 출처 감점', TIER_PENALTY, True, TOP_N)]:
    if name == 'LoRA + 출처 감점' and TIER_PENALTY == 0:
        print('선택된 감점이 0이라 LoRA + 출처 감점은 LoRA와 같습니다. 건너뜀')
        continue
    runs[name] = run(pen, lora, n)
    print(f"{name} 끝 ({runs[name]['sec']:.2f}초/질문)")

In [ ]:
nh = len(test_h)
rows = {}
for name, r in runs.items():
    A, D = r['answers'], r['docs']
    h = range(nh)
    answered = [i for i in h if not is_refusal(A[i])]
    rows[name] = {
        'F1':           np.mean([best(f1, A[i], qs[i]['answers']) for i in h]),
        'ROUGE-L':      np.mean([best(rouge_l, A[i], qs[i]['answers']) for i in h]),
        '출처 표시율':    np.mean([has_cite(A[i]) for i in answered]) if answered else np.nan,
        '잘못 거절':      np.mean([is_refusal(A[i]) for i in h]),
        '근거 R@N':      np.mean([bool(set(qs[i]['gold_pids']) & {d['pid'] for d in D[i]}) for i in h]),
        "근거 중 '기타'":  np.mean([d['tier'] == '기타' for i in h for d in D[i]]),
        '분야 밖 거절률':  np.mean([is_refusal(A[i]) for i in range(nh, len(qs))]),
        '평균 답변 단어':  np.mean([len(A[i].split()) for i in answered]) if answered else np.nan,
        '초/질문':       r['sec'],
    }
table = pd.DataFrame(rows).T.round(3)
display(table)

summary = '\n'.join([
    f"# 05 LoRA + 출처 감점 평가 ({time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))} KST)", '',
    f'- 학습: v1.1 train 건강 질문 {len(feats):,}개 (거절 예시 {n_ref:,}), LoRA r={LORA_R}, {EPOCHS} epoch, {LLM_NAME}',
    f"- 출처 감점: '기타' 등급 재정렬 점수 -{TIER_PENALTY} (검증 R@5 기준으로 선택) / 거절 기준 {THRESH:.2f}",
    f'- 평가: 건강 {nh}개 + 분야 밖 {len(test_o)}개 (04와 같은 질문), 모두 중복 제거 + 거절 기준 적용', '',
    '## 감점 크기 (검증 200개)', '', pen_table.to_markdown(), '',
    '## 시험 결과', '', table.to_markdown(),
])
with open(f'{OUT_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)
save_jsonl([{'qid': q['qid'], 'domain': q['domain'], 'query': q['query'], 'gold_answer': (q.get('answers') or [''])[0],
             **{name: r['answers'][i] for name, r in runs.items()},
             **{f'{name} 문서': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in r['docs'][i]]
                for name, r in runs.items()}}
            for i, q in enumerate(qs)], f'{OUT_DIR}/eval_answers.jsonl')
print('저장:', OUT_DIR)

In [ ]:
# 답변 비교 예시 (번호를 바꿔 보세요. 0~99 건강, 100~ 분야 밖)
i = 0
print('질문:', qs[i]['query'])
print('정답:', (qs[i].get('answers') or ['-'])[0][:200])
for name, r in runs.items():
    print(f'\n[{name}]', r['answers'][i])
    print('   근거:', ', '.join(f"{d['site']}({d['tier']})" for d in r['docs'][i]))

## 8. 드라이브 저장 마무리 + 런타임 종료

LoRA는 드라이브 `lora/adapter/`에 있습니다. 04 챗봇에서 쓰려면 LLM을 불러온 뒤 `PeftModel.from_pretrained(llm, ADAPTER_DIR)` 한 줄을 넣으면 됩니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()